# 06 — 3-Way EDA for Regulatory Agent Case-Study Selection

**Goal:** use the existing RASFF rebuild data to find evidence-rich `origin × category × hazard` combinations that could support the Regulatory Agent benchmark.

This notebook does **not** choose a case study only because it has the largest count. It first finds repeated combinations, then checks time coverage, serious-label share, and the original `hazards`/`subject` text. `Hazard_Type` is an inherited derived field, so it is used only as a broad exploratory grouping and must be validated against original fields before a case is selected.

### Main questions
1. Which origins have enough notifications to support a case study?
2. Within each major origin, which product categories dominate?
3. Within each `origin × category`, which broad hazard types dominate?
4. Which exact original hazards drive the strongest 3-way combinations?
5. Are those combinations repeated across multiple years rather than concentrated in one short period?
6. Which candidates are suitable for a benchmark that can later connect RASFF analysis → regulation retrieval → tool use → agent investigation?


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'rasff_core.py').exists():
    raise FileNotFoundError('Open this notebook from the RASFF_Rebuild folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rasff_core import load_prepared

df = load_prepared()
print(f'Rows: {len(df):,}')
print(f'Date range: {df.date.min()} → {df.date.max()}')
display(df[['reference','date','origin','category','Hazard_Type','hazards','subject','risk_decision']].head())


## 1. Define the exploratory scope

For the benchmark search, start with **food notifications only**. Empty origins are excluded because the main question requires an origin. We keep all risk-decision labels: the goal here is pattern discovery, not serious-label prediction.

`Hazard_Type` is useful for broad grouping but was inherited from the earlier project. Therefore final candidates are always drilled down using original `hazards` and `subject` text.


In [ ]:
eda = df.loc[(df['type'] == 'food') & df['origin'].ne('')].copy()
eda['year'] = eda['date'].dt.year

print(f'Food rows with known origin: {len(eda):,}')
print(f'Unique origins: {eda.origin.nunique():,}')
print(f'Unique categories: {eda.category.nunique():,}')
print(f'Missing original hazards: {eda.hazards.eq("").sum():,}')


## 2. Q1 — Which origins have enough notifications?

This is only a support check. A high count does **not** mean the origin is more unsafe; RASFF is a notification dataset and counts can reflect trade volume, border-control intensity, targeted controls, and reporting practices.


In [ ]:
origin_summary = (eda.groupby('origin')
    .agg(notifications=('reference','nunique'),
         categories=('category','nunique'),
         hazard_types=('Hazard_Type','nunique'),
         years=('year','nunique'))
    .sort_values('notifications', ascending=False))
display(origin_summary.head(20))


## 3. Q2 — For each major origin, which categories dominate?

This answers the question the current dashboard cannot directly answer: **given an origin, what product categories account for its notifications?**


In [ ]:
TOP_ORIGINS = origin_summary.head(10).index

origin_category = (eda[eda.origin.isin(TOP_ORIGINS)]
    .groupby(['origin','category'])
    .agg(notifications=('reference','nunique'))
    .reset_index())
origin_category['origin_total'] = origin_category.groupby('origin')['notifications'].transform('sum')
origin_category['share_within_origin'] = origin_category['notifications'] / origin_category['origin_total']
origin_category = origin_category.sort_values(['origin','notifications'], ascending=[True,False])

top_categories_by_origin = origin_category.groupby('origin', group_keys=False).head(5)
display(top_categories_by_origin)


## 4. Q3 — Within an origin × category, which broad hazard types dominate?

Now add the third dimension. This is the first direct `origin × category × Hazard_Type` cross-check.


In [ ]:
three_way = (eda[eda.origin.isin(TOP_ORIGINS)]
    .groupby(['origin','category','Hazard_Type'])
    .agg(notifications=('reference','nunique'),
         years=('year','nunique'),
         first_date=('date','min'),
         last_date=('date','max'),
         serious_share=('target','mean'))
    .reset_index())

pair_totals = (eda[eda.origin.isin(TOP_ORIGINS)]
    .groupby(['origin','category'])['reference'].nunique()
    .rename('origin_category_total').reset_index())
three_way = three_way.merge(pair_totals, on=['origin','category'], how='left')
three_way['share_within_origin_category'] = three_way['notifications'] / three_way['origin_category_total']
three_way = three_way.sort_values(['notifications','years'], ascending=False)

display(three_way.head(30))


## 5. Q4 — What are the strongest repeated 3-way candidates?

A useful case study should have enough records and preferably appear in several years. The thresholds below are deliberately editable; they are **screening rules**, not scientific cut-offs.


In [ ]:
MIN_NOTIFICATIONS = 20
MIN_YEARS = 3

candidates = three_way.loc[
    (three_way.notifications >= MIN_NOTIFICATIONS) &
    (three_way.years >= MIN_YEARS)
].copy()

display(candidates.head(25))
print(f'Candidate combinations: {len(candidates)}')


## 6. Q5 — Which original hazards actually drive a candidate?

Do **not** select the final case from `Hazard_Type` alone. Choose a row from `candidates`, set the three values below, and inspect the original `hazards` and `subject` fields.


In [ ]:
# Replace these after inspecting the candidates table.
SELECTED_ORIGIN = candidates.iloc[0]['origin'] if len(candidates) else None
SELECTED_CATEGORY = candidates.iloc[0]['category'] if len(candidates) else None
SELECTED_HAZARD_TYPE = candidates.iloc[0]['Hazard_Type'] if len(candidates) else None

print('Selected:', SELECTED_ORIGIN, '|', SELECTED_CATEGORY, '|', SELECTED_HAZARD_TYPE)

subset = eda.loc[
    (eda.origin == SELECTED_ORIGIN) &
    (eda.category == SELECTED_CATEGORY) &
    (eda.Hazard_Type == SELECTED_HAZARD_TYPE)
].copy()

exact_hazards = (subset.loc[subset.hazards.ne('')]
    .groupby('hazards')['reference'].nunique()
    .sort_values(ascending=False)
    .rename('notifications'))
display(exact_hazards.head(20))

display(subset[['reference','date','subject','hazards','risk_decision']]
        .sort_values('date', ascending=False).head(25))


## 7. Q6 — Is the pattern persistent over time?

A large total caused by one exceptional year is less useful for a general benchmark than a recurring pattern. Inspect annual counts before choosing the scenario.


In [ ]:
yearly = (subset.groupby('year')['reference'].nunique()
          .rename('notifications').reset_index())
display(yearly)

ax = yearly.plot(x='year', y='notifications', kind='bar', legend=False, figsize=(8,4), title='Selected combination by year')
ax.set_ylabel('Notifications')
plt.tight_layout()
plt.show()


## 8. Q7 — Compare candidate concentration, not just raw counts

A combination can be interesting because it is common **within a specific origin/category**, even if it is not the largest combination globally. Use both raw count and within-pair share.


In [ ]:
comparison = candidates[[
    'origin','category','Hazard_Type','notifications','years',
    'share_within_origin_category','serious_share','first_date','last_date'
]].copy()

comparison['share_within_origin_category'] = comparison['share_within_origin_category'].round(3)
comparison['serious_share'] = comparison['serious_share'].round(3)
display(comparison.head(30))


## 9. Q8 — Shortlist benchmark case-study candidates

Do not automate the final choice into a single score yet. Review the top candidates and manually judge these dimensions:

- **Data support:** enough notifications and multiple years?
- **Pattern clarity:** a recognizable exact hazard inside the broad hazard type?
- **Regulatory link:** likely to connect to identifiable EU legislation/official guidance?
- **Tool value:** would answering the question benefit from RASFF search or data analysis rather than model memory alone?
- **Agent value:** can the case naturally progress from lookup → evidence retrieval → analysis → multi-step investigation?
- **Interpretability:** can we explain the pattern without implying that notification count equals prevalence or true risk?

Pick **2–3 candidates**, not one, for the next regulatory-source feasibility check.


In [ ]:
shortlist = comparison.head(15).copy()
shortlist['regulatory_link_checked'] = False
shortlist['exact_hazard_validated'] = False
shortlist['notes'] = ''
display(shortlist)

# Optional: save for the next benchmark-design step.
out = ROOT / 'artifacts' / 'regulatory_agent_case_candidates.csv'
shortlist.to_csv(out, index=False)
print('Saved:', out)


## Interpretation guardrails

1. **Notification count ≠ prevalence or risk.** RASFF contains notified events, not all products placed on the EU market or all inspections.
2. **Origin ≠ causation.** Country patterns can be influenced by trade volume, border controls, targeted sampling, and reporting practices.
3. **`Hazard_Type` is derived.** Validate the final scenario with original `hazards` and `subject` text.
4. **Missing hazards matter.** A blank `hazards` field does not mean no hazard existed.
5. **Case-study selection is not model training.** This EDA is for designing a realistic Regulatory Agent benchmark.

### Next step
After running the notebook, bring the top 10–15 rows of `candidates`/`comparison` and the exact-hazard drill-down for 2–3 interesting combinations. We will then check which candidates have enough official EU regulatory evidence before freezing the 10-question benchmark.
